In [2]:
from typing import Sequence
from langchain_core.prompts import PromptTemplate,ChatPromptTemplate,MessagesPlaceholder
from langchain_community.chat_models.tongyi import ChatTongyi
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import BaseMessage,message_to_dict,messages_from_dict
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory,BaseChatMessageHistory
import os
import json

class FileChatMessageHistory(BaseChatMessageHistory):
    def __init__(self,session_id,storage_path):
        self.session_id=session_id
        self.storage_path=storage_path
        self.file_path = os.path.join(self.storage_path, self.session_id)
        # 确保文件夹存在
        os.makedirs(os.path.dirname(self.file_path), exist_ok=True)

    def add_messages(self, messages: Sequence[BaseMessage]):
        all_messages=list(self.messages)
        all_messages.extend(messages)

        # new_messages=[]
        # for message in all_messages:
        #     d=message_to_dict(message=message)
        #     new_messages.append(d)

        new_messages = [message_to_dict(message) for message in all_messages]

        with open(self.file_path, "w", encoding="utf-8") as f:
            json.dump(new_messages, f)

# 将messages方法变成成员属性
    @property
    def messages(self) -> list[BaseMessage]:
        try:
            with open(self.file_path,"r",encoding="utf-8") as f:
                messages=json.load(f)
                return messages_from_dict(messages)
        except FileNotFoundError:
            return []
        
    def clear():
        with open(self.file_path,"r",encoding="utf-8") as f:
            json.dump([], f)
        
        
model=ChatTongyi(model="qwen3-max")

# prompt=PromptTemplate.from_template(
    # "你需要根据会话历史回应问题对话历史{chat_history},用户提问：{input}, 请回答"
# )
prompt=ChatPromptTemplate.from_messages([
    ("system", "你需要根据会话历史回应问题"),
    MessagesPlaceholder("chat_history"),
    ("user","用户提问：{input}, 请回答")
])

def print_prompt(p):
    print("="*10,p,"="*10)
    return p

str_parser=StrOutputParser()

base_chain=prompt |print_prompt | model | str_parser
def get_history(session_id):
    return FileChatMessageHistory(session_id, "./chat_history")

conversation_chain =RunnableWithMessageHistory(
    base_chain,
    get_history,
    input_messages_key="input",
    history_messages_key="chat_history"
)

if __name__ == "__main__":
    session_config={
        "configurable":{
            "session_id":"user001"
        }
    }
    # res=conversation_chain.invoke({
    #     "input":"peter有一个猫",
    # }, session_config)

    # print(res)
    # res=conversation_chain.invoke({
    #     "input":"lisa有一个狗",
    # }, session_config)

    # print(res)
    res=conversation_chain.invoke({
        "input":"一共几个宠物",
    }, session_config)

    print(res)

========== messages=[SystemMessage(content='你需要根据会话历史回应问题', additional_kwargs={}, response_metadata={}), HumanMessage(content='peter有一个猫', additional_kwargs={}, response_metadata={}), AIMessage(content='Peter有一只猫。', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='lisa有一个狗', additional_kwargs={}, response_metadata={}), AIMessage(content='Lisa有一只狗。', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='一共几个宠物', additional_kwargs={}, response_metadata={}), AIMessage(content='一共2个宠物：Peter有一只猫，Lisa有一只狗。', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='用户提问：一共几个宠物, 请回答', additional_kwargs={}, response_metadata={})] ==========
一共2个宠物。
